# 01_cuDF / 01 — cuDF basics

**cuDF** is a GPU DataFrame library with a pandas-like API. Data lives in GPU memory (in Apache Arrow columnar format), and operations run as CUDA kernels.

This notebook covers:
1. Environment check (GPU, versions)
2. `Series` and `DataFrame` — creating, selecting, moving between pandas and cuDF
3. dtypes — numeric, strings, categoricals, datetimes, lists, structs, and **nulls**
4. IO — CSV and Parquet, with a first pandas vs cuDF timing
5. GPU memory — how much a DataFrame uses and how to free it

**Hardware:** NVIDIA T4 (16 GB). Works on Google Colab / Kaggle T4 runtimes.

## 1. Environment check

In [ ]:
# Install cuDF only if it is missing (Colab GPU runtimes usually ship it already).
# If this installs something, restart the kernel before continuing.
import importlib.util, subprocess, sys

if importlib.util.find_spec("cudf") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "cudf-cu12", "--extra-index-url=https://pypi.nvidia.com"])
    print("cuDF installed: restart the kernel now.")
else:
    print("cuDF already installed.")

In [ ]:
!nvidia-smi

In [ ]:
import platform
import cudf, cupy as cp, numpy as np, pandas as pd

props = cp.cuda.runtime.getDeviceProperties(0)
free_b, total_b = cp.cuda.runtime.memGetInfo()

print(f"Python  : {platform.python_version()}")
print(f"cuDF    : {cudf.__version__}")
print(f"CuPy    : {cp.__version__}")
print(f"pandas  : {pd.__version__}")
print(f"NumPy   : {np.__version__}")
print(f"GPU     : {props['name'].decode()}  (compute capability {props['major']}.{props['minor']})")
print(f"GPU mem : {free_b / 1e9:.1f} GB free of {total_b / 1e9:.1f} GB")

### A small timing helper

GPU work is **asynchronous**: Python can get control back before the kernel has finished. To time GPU code correctly, we synchronize the device before stopping the clock, run a warmup first (the first call pays for kernel compilation and memory allocation), and take the median of several runs.

In [ ]:
import time, statistics

def gpu_sync():
    cp.cuda.Device().synchronize()

def timeit(fn, repeats=5, warmup=1):
    """Median wall time in seconds, with GPU synchronized before each stop."""
    for _ in range(warmup):
        fn(); gpu_sync()
    times = []
    for _ in range(repeats):
        gpu_sync()
        t0 = time.perf_counter()
        fn()
        gpu_sync()
        times.append(time.perf_counter() - t0)
    return statistics.median(times)

## 2. Series and DataFrame

The API mirrors pandas: most code only needs `pd.` swapped for `cudf.`.

In [ ]:
s = cudf.Series([10, 20, 30, None, 50], name="values")
s

In [ ]:
print("sum  :", s.sum())       # nulls are skipped
print("mean :", s.mean())
print("nulls:", s.isna().sum())
s * 2 + 1

In [ ]:
gdf = cudf.DataFrame({
    "id":    [1, 2, 3, 4, 5, 6],
    "city":  ["Hyderabad", "Pune", "Jaipur", "Pune", "Hyderabad", "Jaipur"],
    "sales": [120.5, 98.0, 143.2, 87.1, 160.0, 110.4],
    "units": [12, 9, 15, 8, 17, 11],
})
gdf

In [ ]:
print(type(gdf))
gdf.info()

### Selecting rows and columns

In [ ]:
gdf[["city", "sales"]].head(3)

In [ ]:
gdf.loc[gdf["sales"] > 100, ["id", "city", "sales"]]

In [ ]:
gdf.iloc[1:4]

In [ ]:
gdf.query("units >= 10 and city == 'Hyderabad'")

In [ ]:
# New columns, sorting, simple aggregation
gdf["price_per_unit"] = gdf["sales"] / gdf["units"]
gdf.sort_values("price_per_unit", ascending=False)

In [ ]:
gdf.describe()

### Moving between pandas, cuDF and CuPy

Every conversion copies data between host (CPU) RAM and GPU memory over PCIe. That copy is often the slowest step, so keep data on the GPU for as long as possible.

In [ ]:
pdf = gdf.to_pandas()              # GPU -> CPU
print(type(pdf))

gdf2 = cudf.from_pandas(pdf)       # CPU -> GPU
print(type(gdf2))

arr = gdf[["sales", "units"]].to_cupy()   # stays on the GPU, zero CPU copies
print(type(arr), arr.shape, arr.dtype)

## 3. dtypes

cuDF uses **Arrow-style nulls**: every column can hold `<NA>`, including ints and strings. pandas traditionally uses `NaN` for missing values, which forces int columns to float. This is one of the most common differences you will see.

In [ ]:
# Integers with a missing value: pandas upcasts to float64, cuDF keeps int64 + null
p_int = pd.Series([1, 2, None])
g_int = cudf.Series([1, 2, None])
print("pandas:", p_int.dtype, list(p_int))
print("cuDF  :", g_int.dtype, g_int.to_arrow().to_pylist())

In [ ]:
# NaN vs null: by default a float NaN coming from pandas can be kept as NaN or turned into null
p_float = pd.Series([1.0, np.nan, 3.0])
print("nan_as_null=True :", cudf.Series.from_pandas(p_float, nan_as_null=True).isna().sum(), "null(s)")
print("nan_as_null=False:", cudf.Series.from_pandas(p_float, nan_as_null=False).isna().sum(), "null(s)")

### Strings

In [ ]:
names = cudf.Series(["  Alice ", "bob", None, "CHARLIE", "dave_smith"])
cudf.DataFrame({
    "raw":      names,
    "clean":    names.str.strip().str.lower(),
    "length":   names.str.len(),
    "has_a":    names.str.contains("a", regex=False),
    "first":    names.str.split("_").list.get(0),
})

### Categoricals

In [ ]:
cities = gdf["city"].astype("category")
print("categories:", cities.cat.categories.to_pandas().tolist())
print("codes     :", cities.cat.codes.to_pandas().tolist())
print("memory (string)  :", gdf["city"].memory_usage(), "bytes")
print("memory (category):", cities.memory_usage(), "bytes")

### Datetimes

In [ ]:
ts = cudf.Series(pd.date_range("2026-01-01", periods=6, freq="17D"))
cudf.DataFrame({
    "ts":      ts,
    "year":    ts.dt.year,
    "month":   ts.dt.month,
    "weekday": ts.dt.weekday,
    "plus_1d": ts + pd.Timedelta(days=1),
})

### Nested types: lists and structs

Arrow supports nested columns natively, so cuDF can hold a list or a dict in every row, all on the GPU.

In [ ]:
tags = cudf.Series([["gpu", "rapids"], ["cuda"], None, ["pandas", "arrow", "parquet"]])
print(tags.dtype)
cudf.DataFrame({"tags": tags, "n_tags": tags.list.len(), "first": tags.list.get(0)})

In [ ]:
people = cudf.Series([{"name": "Asha", "age": 31}, {"name": "Ravi", "age": 27}, None])
print(people.dtype)
cudf.DataFrame({"name": people.struct.field("name"), "age": people.struct.field("age")})

### Casting

In [ ]:
mixed = cudf.DataFrame({"a": ["1", "2", "3"], "b": [1.7, 2.2, 3.9]})
mixed["a"] = mixed["a"].astype("int32")
mixed["b"] = mixed["b"].astype("int16")      # truncates toward zero
mixed.dtypes

## 4. IO: CSV and Parquet

We generate a synthetic table, write it to disk, then read it back with pandas and with cuDF.

- **CSV** is text: every value has to be parsed. cuDF parses in parallel on the GPU.
- **Parquet** is columnar and compressed, and cuDF decodes it directly on the GPU. It is the preferred format for GPU work.

`N_ROWS` is sized for a T4. Increase it to see the gap grow.

In [ ]:
import os

N_ROWS = 5_000_000
DATA_DIR = "data"
os.makedirs(DATA_DIR, exist_ok=True)

rng = cp.random.default_rng(42)
big = cudf.DataFrame({
    "user_id":  rng.integers(0, 1_000_000, N_ROWS),
    "amount":   rng.random(N_ROWS) * 1000,
    "quantity": rng.integers(1, 20, N_ROWS).astype("int32"),
    "category": cudf.Series([f"cat_{i}" for i in range(50)])
                    .take(rng.integers(0, 50, N_ROWS)).reset_index(drop=True),
})
print(f"{len(big):,} rows")
big.head()

In [ ]:
csv_path = os.path.join(DATA_DIR, "transactions.csv")
pq_path  = os.path.join(DATA_DIR, "transactions.parquet")

t_csv = timeit(lambda: big.to_csv(csv_path, index=False), repeats=1, warmup=0)
t_pq  = timeit(lambda: big.to_parquet(pq_path), repeats=1, warmup=0)
print(f"cuDF write CSV     : {t_csv:.2f} s  ({os.path.getsize(csv_path) / 1e6:.0f} MB)")
print(f"cuDF write Parquet : {t_pq:.2f} s  ({os.path.getsize(pq_path) / 1e6:.0f} MB)")

In [ ]:
results = {
    ("CSV", "pandas"):     timeit(lambda: pd.read_csv(csv_path), repeats=3),
    ("CSV", "cuDF"):       timeit(lambda: cudf.read_csv(csv_path), repeats=3),
    ("Parquet", "pandas"): timeit(lambda: pd.read_parquet(pq_path), repeats=3),
    ("Parquet", "cuDF"):   timeit(lambda: cudf.read_parquet(pq_path), repeats=3),
}

read_times = pd.Series(results).unstack()
read_times["speedup"] = read_times["pandas"] / read_times["cuDF"]
read_times.round(3)

In [ ]:
import matplotlib.pyplot as plt

ax = read_times[["pandas", "cuDF"]].plot.bar(rot=0, figsize=(6, 3.5))
ax.set_ylabel("seconds (median, lower is better)")
ax.set_title(f"Read {N_ROWS:,} rows")
for c in ax.containers:
    ax.bar_label(c, fmt="%.2f", fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# Read only what you need: column projection skips the other columns entirely
cols = cudf.read_parquet(pq_path, columns=["user_id", "amount"])
cols.dtypes

## 5. GPU memory

A T4 has 16 GB. A useful rule of thumb is that a workflow needs **2–3× the size of the data** in GPU memory, because many operations make temporary copies.

In [ ]:
def gpu_mem(label=""):
    free_b, total_b = cp.cuda.runtime.memGetInfo()
    print(f"{label:<28} used {(total_b - free_b) / 1e9:5.2f} GB / {total_b / 1e9:.1f} GB")

gpu_mem("now")

In [ ]:
usage = big.memory_usage(deep=True).to_pandas()
print((usage / 1e6).round(1).astype(str) + " MB")
print(f"total: {usage.sum() / 1e6:.0f} MB")

In [ ]:
# Strings are stored as one big character buffer + offsets; categoricals store small int codes
before = big["category"].memory_usage()
big["category"] = big["category"].astype("category")
after = big["category"].memory_usage()
print(f"category column: {before / 1e6:.0f} MB -> {after / 1e6:.0f} MB")

In [ ]:
# Downcasting numeric columns also saves memory
before = big.memory_usage(deep=True).sum()
big["user_id"] = big["user_id"].astype("int32")
big["amount"]  = big["amount"].astype("float32")
after = big.memory_usage(deep=True).sum()
print(f"whole frame: {before / 1e6:.0f} MB -> {after / 1e6:.0f} MB")

In [ ]:
# Freeing memory: drop every reference, then let Python collect it
import gc

gpu_mem("before del")
del big, cols
gc.collect()
gpu_mem("after del + gc.collect()")

## 6. Host ↔ device transfer cost

Copying data between CPU and GPU goes over PCIe. On small or simple workloads this copy can cost more than the computation saves.

In [ ]:
pdf_big = pd.read_parquet(pq_path)

t_h2d = timeit(lambda: cudf.from_pandas(pdf_big), repeats=3)
gdf_big = cudf.from_pandas(pdf_big)
t_d2h = timeit(lambda: gdf_big.to_pandas(), repeats=3)
t_sum_gpu = timeit(lambda: gdf_big["amount"].sum(), repeats=5)
t_sum_cpu = timeit(lambda: pdf_big["amount"].sum(), repeats=5)

print(f"CPU -> GPU copy : {t_h2d * 1e3:8.1f} ms")
print(f"GPU -> CPU copy : {t_d2h * 1e3:8.1f} ms")
print(f"sum on GPU      : {t_sum_gpu * 1e3:8.2f} ms")
print(f"sum on CPU      : {t_sum_cpu * 1e3:8.2f} ms")

A single `sum()` is far cheaper than copying the data to the GPU. The GPU pays off when you load directly onto it (e.g. `cudf.read_parquet`) and run **many** operations before bringing a small result back.

In [ ]:
del pdf_big, gdf_big
gc.collect()
gpu_mem("end of notebook")

## Summary

- cuDF mirrors the pandas API: `cudf.DataFrame`, `.loc`, `.query`, `.sort_values`, `.describe`, `.str`, `.dt`.
- Every column is nullable (`<NA>`); ints stay ints when values are missing. Watch `nan_as_null` when converting from pandas.
- Nested `list` and `struct` columns are first-class.
- Prefer **Parquet** and read only the columns you need.
- Measure GPU time with a sync and a warmup.
- Keep data on the GPU: PCIe copies are expensive compared to most single operations.
- Use categoricals and smaller numeric dtypes to fit more data on a 16 GB T4.

**Try it:** raise `N_ROWS` to 20M and rerun section 4. Where does pandas start to struggle?

**Next:** `02_cudf_operations.ipynb` — groupby, joins, sorting, string ops and window functions.